# 00 — Environment check
Validate configuration, Unity Catalog schemas, and Volume paths before a pipeline run.

In [ ]:
from pathlib import Path
import sys

source_root = next((root / "src" for root in (Path.cwd(), *Path.cwd().parents) if (root / "src").is_dir()), None)
if source_root is not None and str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

In [ ]:
ENVIRONMENT = "dev"
try:
    dbutils.widgets.text("environment", ENVIRONMENT)
    ENVIRONMENT = dbutils.widgets.get("environment")
except NameError:
    pass
print(f"Selected environment: {ENVIRONMENT}")

In [ ]:
from finops_cloud.audit.snapshots import ensure_audit_tables
from finops_cloud.config import load_config
from finops_cloud.runtime import ensure_schemas, get_spark

config = load_config(ENVIRONMENT)
spark_session = get_spark(config.profile)
ensure_schemas(spark_session, config)
ensure_audit_tables(spark_session, config)
print({
    "catalog": config.catalog,
    "raw_schema": f"{config.raw_catalog}.{config.raw_schema}",
    "operations_schema": config.schema("ops"),
    "source_volume": config.source_volume,
    "archive_volume": config.archive_volume,
    "gcs_bucket": config.gcs_bucket,
})

In [ ]:
display(spark_session.sql(f"SHOW SCHEMAS IN `{config.catalog}`"))
display(spark_session.sql(f"SHOW VOLUMES IN `{config.raw_catalog}`.`{config.raw_schema}`"))
display(spark_session.sql(f"SHOW TABLES IN `{config.operations_catalog}`.`{config.operations_schema}`"))
display(spark_session.sql(f"LIST '{config.source_volume}/monthly'"))